# Profile Interpolation Analysis

Compares the raw in-situ effective radius profiles (17–103 altitude levels, variable per file)
against the 10-level interpolated versions that will be used as training targets.

Reads directly from the `.mat` files — no HDF5 required.

**Profile convention throughout**: index 0 = cloud top (highest altitude), index -1 = cloud base.

In [12]:
import numpy as np
import scipy.io
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from pathlib import Path

In [13]:
# ── Configuration ──────────────────────────────────────────────────────────────
MAT_DIR   = Path('/Volumes/My Passport/neural_network_training_data/saz0_allProfiles/')
N_LEVELS  = 10   # number of interpolated levels used as training target

mat_files = sorted(f for f in MAT_DIR.glob('*.mat')
                   if not f.name.startswith('._'))
print(f'Found {len(mat_files)} .mat files')

Found 0 .mat files


In [14]:
def interpolate_profile(re_raw, z_raw, n_levels_out):
    """
    Interpolate a variable-length r_e profile to n_levels_out evenly-spaced
    altitude levels using the actual altitude coordinates in z_raw.

    Parameters
    ----------
    re_raw : (n,)  effective radius, cloud top → base (μm)
    z_raw  : (n,)  altitude, cloud top → base (km, decreasing)
    n_levels_out : int

    Returns
    -------
    re_interp : (n_levels_out,)  — cloud top → base
    z_interp  : (n_levels_out,)  — evenly-spaced altitudes, cloud top → base (km)
    """
    z_asc  = z_raw[::-1]   # ascending: cloud base → top
    re_asc = re_raw[::-1]
    z_target_asc = np.linspace(z_asc[0], z_asc[-1], n_levels_out)
    re_interp_asc = np.interp(z_target_asc, z_asc, re_asc)
    return re_interp_asc[::-1], z_target_asc[::-1]  # flip back: top → base


def interpolate_profile_tau_weighted(
    re_raw,
    tau_raw,
    z_raw=None,
    n_top=6,
    n_bot=4,
    split_tau=0.4,
):
    """
    Interpolate an in-situ r_e profile onto a non-uniformly spaced
    optical-depth grid that places more levels near cloud top, where
    reflected radiance sensitivity is highest.

    The normalized optical depth τ̃ spans [0, 1] (cloud top → cloud base).
    n_top points are placed evenly over [0, split_tau] and n_bot points over
    (split_tau, 1], giving n_top + n_bot levels total.

    Default parameters (n_top=6, n_bot=4, split_tau=0.4) produce a 10-level
    grid that concentrates half the levels in the top 40 % of optical depth.

    Parameters
    ----------
    re_raw    : (n,)       effective radius, cloud top → base (μm)
    tau_raw   : (n,)       cumulative optical depth, cloud top → base;
                           may be ascending (descending flight) or descending
                           (ascending flight) — handled automatically
    z_raw     : (n,) or None  altitude in km, cloud top → base (decreasing)
    n_top     : int        levels in [0, split_tau]        (default 6)
    n_bot     : int        levels in (split_tau, 1]        (default 4)
    split_tau : float      normalized τ split point        (default 0.4)

    Returns
    -------
    re_interp : (n_top + n_bot,) float32
        Effective radius at each sampled level (cloud top → base order).
    tau_grid  : (n_top + n_bot,) float64
        Normalized optical depth τ̃ ∈ [0, 1] at each level.
        Use as the y-axis when plotting in τ-space (invert axis so 0 is at top).
    z_interp  : (n_top + n_bot,) float64 or None
        Altitude (km) at each sampled level.
        Use as the y-axis when plotting in z-space (altitude decreases top → base).

    Plotting guidance
    -----------------
    z-space  : ax.plot(re_interp, z_interp)
    τ-space  : ax.plot(re_interp, tau_grid); ax.invert_yaxis()
               (τ̃=0 at cloud top → put at top of the plot)
    """
    # Coerce to 1-D float64 so indexing always works.
    tau_raw = np.atleast_1d(np.asarray(tau_raw, dtype=np.float64))
    re_raw  = np.atleast_1d(np.asarray(re_raw,  dtype=np.float64))

    # Normalize to [0, 1] using global min/max so the result is correct
    # regardless of whether tau was stored cloud-top→base (ascending) or
    # cloud-base→top (descending), and robust to slight non-monotonicities.
    tau_min, tau_max = tau_raw.min(), tau_raw.max()
    tau_norm = (tau_raw - tau_min) / (tau_max - tau_min)

    # Sort by ascending τ̃ so np.interp receives a valid xp array.
    # For profiles already in top→base (ascending τ) order this is a no-op.
    order      = np.argsort(tau_norm)
    tau_sorted = tau_norm[order]
    re_sorted  = re_raw[order]
    z_sorted   = np.asarray(z_raw, dtype=np.float64)[order] if z_raw is not None else None

    # Build non-uniform grid
    tau_top  = np.linspace(0.0, split_tau, n_top)
    tau_bot  = np.linspace(split_tau, 1.0, n_bot + 1)[1:]
    tau_grid = np.concatenate([tau_top, tau_bot])

    re_interp = np.interp(tau_grid, tau_sorted, re_sorted).astype(np.float32)
    z_interp  = np.interp(tau_grid, tau_sorted, z_sorted) if z_sorted is not None else None

    return re_interp, tau_grid, z_interp


In [15]:
# ── Load all profiles ──────────────────────────────────────────────────────────
profiles_raw  = []   # list of (n_i,) arrays, variable length
altitudes_raw = []   # corresponding z arrays (km, top → base)
tau_raw_all   = []   # cumulative optical depth arrays (top → base)

profiles_10   = []   # (N_LEVELS,) evenly-spaced altitude interpolation
altitudes_10  = []   # (N_LEVELS,) evenly-spaced altitude grid (km)

profiles_tau  = []   # (N_LEVELS,) τ-weighted non-uniform interpolation
altitudes_tau = []   # (N_LEVELS,) altitude at each τ-weighted level (km)
tau_grids     = []   # (N_LEVELS,) normalized τ values of the sampled levels
tau_c_all     = []

for path in mat_files:
    d = scipy.io.loadmat(path, squeeze_me=True)
    re  = d['re'][()].astype(np.float64)
    z   = d['z'][()].astype(np.float64)
    tau = d['tau'][()].astype(np.float64)   # cumulative OD, cloud top → base
    tau_c = float(tau.max())

    re_10,  z_10  = interpolate_profile(re, z, N_LEVELS)
    re_tau, tau_grid, z_tau = interpolate_profile_tau_weighted(re, tau, z_raw=z)

    profiles_raw.append(re)
    altitudes_raw.append(z)
    tau_raw_all.append(tau)

    profiles_10.append(re_10)
    altitudes_10.append(z_10)

    profiles_tau.append(re_tau)
    altitudes_tau.append(z_tau)
    tau_grids.append(tau_grid)

    tau_c_all.append(tau_c)

n_profiles    = len(profiles_raw)
n_levels_each = np.array([len(r) for r in profiles_raw])
tau_c_all     = np.array(tau_c_all)

print(f'Loaded {n_profiles} profiles')
print(f'Profile lengths: min={n_levels_each.min()}, max={n_levels_each.max()}, mean={n_levels_each.mean():.1f}')
print(f'tau_c range: [{tau_c_all.min():.1f}, {tau_c_all.max():.1f}]')
print(f'\nτ-weighted grid (shared across all profiles):')
print(f'  τ̃ = {tau_grids[0].round(3)}')


Loaded 0 profiles


ValueError: zero-size array to reduction operation minimum which has no identity

## 1. Profile length distribution

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3.5))
ax.hist(n_levels_each, bins=20, color='steelblue', edgecolor='white', linewidth=0.5)
ax.axvline(N_LEVELS, color='firebrick', linewidth=1.5, linestyle='--',
           label=f'Training target ({N_LEVELS} levels)')
ax.set_xlabel('Number of in-situ altitude levels')
ax.set_ylabel('Number of profiles')
ax.set_title('Distribution of raw profile lengths')
ax.legend()
fig.tight_layout()
plt.show()

## 2. Individual profile comparisons

Gray line with dots: raw in-situ measurement at full resolution.  
Colored line with filled circles: 10-level interpolated version (the training target).  
Profiles are sorted by number of raw levels (fewest → most) so you can see how interpolation quality scales with resolution.

In [ ]:
# ── Plotting options ───────────────────────────────────────────────────────────
# 'z'   — y-axis is altitude (km); cloud top at top of plot (higher km value)
# 'tau' — y-axis is normalized optical depth τ̃ ∈ [0,1]; τ̃=0 (cloud top) at top
PLOT_SPACE = 'z'

# Sort profiles by number of raw levels so the grid shows a range of complexities
sort_idx = np.argsort(n_levels_each)

# Pick N_SHOW profiles spaced evenly through the sorted list
N_SHOW = 12
show_idx = sort_idx[np.linspace(0, n_profiles - 1, N_SHOW, dtype=int)]

ncols = 4
nrows = int(np.ceil(N_SHOW / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 3.0, nrows * 3.5),
                         sharey=False, sharex=False)
axes = axes.flatten()

for ax, idx in zip(axes, show_idx):
    re_raw  = profiles_raw[idx]
    z_raw   = altitudes_raw[idx]
    re_10   = profiles_10[idx]
    z_10    = altitudes_10[idx]
    re_tau  = profiles_tau[idx]
    z_tau   = altitudes_tau[idx]
    tau_i   = tau_raw_all[idx]

    if PLOT_SPACE == 'z':
        y_raw      = z_raw
        y_10       = z_10
        y_tau_plot = z_tau
        ylabel     = 'Altitude (km)'
        invert_y   = False
    else:  # 'tau'
        # Normalize raw tau to [0, 1] for this profile
        y_raw = (tau_i - tau_i.min()) / (tau_i.max() - tau_i.min())
        # Estimate normalized tau at even-z levels via interpolation
        z_asc_i       = z_raw[::-1]
        tau_norm_asc  = y_raw[::-1]
        y_10          = np.interp(z_10[::-1], z_asc_i, tau_norm_asc)[::-1]
        # τ-weighted levels have exact tau by construction
        y_tau_plot    = tau_grids[idx]
        ylabel        = 'Normalized optical depth τ̃  (0 = cloud top)'
        invert_y      = True

    # Raw profile
    ax.plot(re_raw, y_raw, color='0.55', linewidth=1.2,
            marker='o', markersize=2.5, label='Raw in-situ')

    # Evenly-spaced altitude interpolation
    ax.plot(re_10, y_10, color='steelblue', linewidth=1.8,
            marker='o', markersize=5, label=f'{N_LEVELS}-level (even z)')

    # τ-weighted non-uniform interpolation
    ax.plot(re_tau, y_tau_plot, color='darkorange', linewidth=1.8,
            marker='s', markersize=5, linestyle='--',
            label=f'{N_LEVELS}-level (τ-weighted)')

    if invert_y:
        ax.invert_yaxis()

    n_lev = n_levels_each[idx]
    ax.set_title(f'Profile {idx+1}  |  {n_lev} levels  |  τ={tau_c_all[idx]:.1f}',
                 fontsize=8)
    ax.set_xlabel('$r_e$ (μm)', fontsize=8)
    ax.set_ylabel(ylabel, fontsize=8)
    ax.tick_params(labelsize=7)

axes[0].legend(fontsize=7, loc='lower right')

for ax in axes[N_SHOW:]:
    ax.set_visible(False)

space_label = 'altitude' if PLOT_SPACE == 'z' else 'normalized optical depth τ̃'
fig.suptitle(f'Raw in-situ vs. even-z vs. τ-weighted 10-level profiles\n'
             f'(y-axis: {space_label}, sorted by number of raw levels fewest → most)',
             fontsize=10, y=1.01)
fig.tight_layout()
plt.show()


## 3. Interpolation error quantification

To measure how well the 10-level profile represents the original, we upsample the 10-level
version back onto the original altitude grid and compare to the raw measurements.  
This gives a per-profile root-mean-squared error (RMSE) and maximum absolute error.

In [ ]:
rmse_all     = np.zeros(n_profiles)
mae_all      = np.zeros(n_profiles)
maxerr_all   = np.zeros(n_profiles)
rmse_tau_all   = np.zeros(n_profiles)
mae_tau_all    = np.zeros(n_profiles)
maxerr_tau_all = np.zeros(n_profiles)

for i in range(n_profiles):
    re_raw = profiles_raw[i]
    z_raw  = altitudes_raw[i]

    # ── Even-z interpolation error ────────────────────────────────────────────
    re_10 = profiles_10[i]
    z_10  = altitudes_10[i]
    z_10_asc   = z_10[::-1]
    re_10_asc  = re_10[::-1]
    z_raw_asc  = z_raw[::-1]
    re_recon_asc = np.interp(z_raw_asc, z_10_asc, re_10_asc)
    re_recon = re_recon_asc[::-1]
    err = re_recon - re_raw
    rmse_all[i]   = np.sqrt(np.mean(err**2))
    mae_all[i]    = np.mean(np.abs(err))
    maxerr_all[i] = np.max(np.abs(err))

    # ── τ-weighted interpolation error ────────────────────────────────────────
    re_tau = profiles_tau[i]
    z_tau  = altitudes_tau[i]
    z_tau_asc  = z_tau[::-1]
    re_tau_asc = re_tau[::-1]
    re_recon_tau_asc = np.interp(z_raw_asc, z_tau_asc, re_tau_asc)
    re_recon_tau = re_recon_tau_asc[::-1]
    err_tau = re_recon_tau - re_raw
    rmse_tau_all[i]   = np.sqrt(np.mean(err_tau**2))
    mae_tau_all[i]    = np.mean(np.abs(err_tau))
    maxerr_tau_all[i] = np.max(np.abs(err_tau))

print(f'Interpolation error (reconstructed on raw grid vs. raw):')
print(f'  {"Method":<28} {"RMSE mean":>10} {"RMSE max":>10} {"MAE mean":>10} {"Max err max":>12}')
print(f'  {"-"*72}')
print(f'  {"Even-z (altitude grid)":<28} {rmse_all.mean():>10.3f} {rmse_all.max():>10.3f} {mae_all.mean():>10.3f} {maxerr_all.max():>12.3f} μm')
print(f'  {"τ-weighted (OD grid)":<28} {rmse_tau_all.mean():>10.3f} {rmse_tau_all.max():>10.3f} {mae_tau_all.mean():>10.3f} {maxerr_tau_all.max():>12.3f} μm')


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 3.5))

# RMSE vs. number of raw levels — both methods
axes[0].scatter(n_levels_each, rmse_all, c=tau_c_all,
                cmap='viridis', s=30, alpha=0.7, marker='o', label='Even-z')
sc = axes[0].scatter(n_levels_each, rmse_tau_all, c=tau_c_all,
                     cmap='viridis', s=30, alpha=0.7, marker='s', label='τ-weighted')
plt.colorbar(sc, ax=axes[0], label='τ_c')
axes[0].set_xlabel('Number of raw levels')
axes[0].set_ylabel('RMSE (μm)')
axes[0].set_title('RMSE vs. profile resolution')
axes[0].legend(fontsize=8)

# Max absolute error vs. number of raw levels
axes[1].scatter(n_levels_each, maxerr_all, c=tau_c_all,
                cmap='viridis', s=30, alpha=0.7, marker='o', label='Even-z')
sc2 = axes[1].scatter(n_levels_each, maxerr_tau_all, c=tau_c_all,
                      cmap='viridis', s=30, alpha=0.7, marker='s', label='τ-weighted')
plt.colorbar(sc2, ax=axes[1], label='τ_c')
axes[1].set_xlabel('Number of raw levels')
axes[1].set_ylabel('Max absolute error (μm)')
axes[1].set_title('Max error vs. profile resolution')
axes[1].legend(fontsize=8)

# Error distributions side-by-side
axes[2].hist(rmse_all,     bins=15, alpha=0.65, label='RMSE  (even-z)',      color='steelblue')
axes[2].hist(rmse_tau_all, bins=15, alpha=0.65, label='RMSE  (τ-weighted)',   color='darkorange')
axes[2].hist(maxerr_all,     bins=15, alpha=0.40, label='Max |err| (even-z)',    color='steelblue',   linestyle='--', edgecolor='steelblue', histtype='step', linewidth=1.5)
axes[2].hist(maxerr_tau_all, bins=15, alpha=0.40, label='Max |err| (τ-weighted)', color='darkorange', linestyle='--', edgecolor='darkorange', histtype='step', linewidth=1.5)
axes[2].set_xlabel('Error (μm)')
axes[2].set_ylabel('Count')
axes[2].set_title('Error distributions: even-z vs. τ-weighted')
axes[2].legend(fontsize=7)

fig.suptitle('Interpolation error comparison: even altitude grid vs. τ-weighted grid',
             fontsize=10)
fig.tight_layout()
plt.show()


## 4. All profiles — overview

Raw profiles (gray) and 10-level interpolated profiles (colored by τ_c) plotted together.
Altitude is normalized to cloud fraction (0 = cloud base, 1 = cloud top)
so that profiles of different geometric thickness can be overlaid.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 5), sharey=False)

cmap  = plt.get_cmap('plasma')
tau_norm = (tau_c_all - tau_c_all.min()) / (tau_c_all.max() - tau_c_all.min())

# Left: actual altitude (km) — profiles will not overlap perfectly because
# clouds are at different altitudes
for i in range(n_profiles):
    axes[0].plot(profiles_raw[i], altitudes_raw[i],
                 color='0.7', linewidth=0.6, alpha=0.5)
for i in range(n_profiles):
    axes[0].plot(profiles_10[i], altitudes_10[i],
                 color=cmap(tau_norm[i]), linewidth=1.2,
                 marker='o', markersize=3, alpha=0.7)

sm = plt.cm.ScalarMappable(cmap='plasma',
     norm=plt.Normalize(tau_c_all.min(), tau_c_all.max()))
plt.colorbar(sm, ax=axes[0], label='τ_c')
axes[0].set_xlabel('$r_e$ (μm)')
axes[0].set_ylabel('Altitude (km)')
axes[0].set_title('All profiles — actual altitude\n(gray = raw, colored = 10-level interp.)')

# Right: normalized altitude (cloud fraction), so all profiles overlay
for i in range(n_profiles):
    z = altitudes_raw[i]
    z_frac = (z - z[-1]) / (z[0] - z[-1])   # 0 at base, 1 at top
    axes[1].plot(profiles_raw[i], z_frac,
                 color='0.7', linewidth=0.6, alpha=0.5)

for i in range(n_profiles):
    z10 = altitudes_10[i]
    z10_frac = (z10 - z10[-1]) / (z10[0] - z10[-1])
    axes[1].plot(profiles_10[i], z10_frac,
                 color=cmap(tau_norm[i]), linewidth=1.2,
                 marker='o', markersize=3, alpha=0.7)

plt.colorbar(sm, ax=axes[1], label='τ_c')
axes[1].set_xlabel('$r_e$ (μm)')
axes[1].set_ylabel('Normalized altitude (0 = base, 1 = top)')
axes[1].set_title('All profiles — normalized altitude\n(gray = raw, colored = 10-level interp.)')

fig.tight_layout()
plt.show()

## 5. Worst-case profiles

Show the profiles where interpolation error (RMSE) is highest,
so you can inspect whether the 10-level representation misses important structure.

In [ ]:
# ── Plotting options ───────────────────────────────────────────────────────────
# 'z'   — y-axis is altitude (km)
# 'tau' — y-axis is normalized optical depth τ̃ ∈ [0,1]; τ̃=0 (cloud top) at top
PLOT_SPACE = 'z'

N_WORST = 6
# Worst cases for the even-z method (highest RMSE)
worst_idx = np.argsort(rmse_all)[-N_WORST:][::-1]

ncols = 3
nrows = int(np.ceil(N_WORST / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 3.2, nrows * 3.8))
axes = axes.flatten()

for ax, idx in zip(axes, worst_idx):
    re_raw  = profiles_raw[idx]
    z_raw   = altitudes_raw[idx]
    re_10   = profiles_10[idx]
    z_10    = altitudes_10[idx]
    re_tau  = profiles_tau[idx]
    z_tau   = altitudes_tau[idx]
    tau_i   = tau_raw_all[idx]

    z_raw_asc = z_raw[::-1]

    # ── Choose y-axis coordinates ─────────────────────────────────────────────
    if PLOT_SPACE == 'z':
        y_raw      = z_raw
        y_10       = z_10
        y_tau_plot = z_tau
        y_raw_asc  = z_raw_asc
        ylabel     = 'Altitude (km)'
        invert_y   = False
    else:  # 'tau'
        tau_norm_i   = (tau_i - tau_i.min()) / (tau_i.max() - tau_i.min())
        y_raw        = tau_norm_i
        tau_norm_asc = tau_norm_i[::-1]
        y_10         = np.interp(z_10[::-1], z_raw_asc, tau_norm_asc)[::-1]
        y_tau_plot   = tau_grids[idx]
        y_raw_asc    = tau_norm_asc
        ylabel       = 'Normalized optical depth τ̃  (0 = cloud top)'
        invert_y     = True

    # ── Reconstruct even-z and τ-weighted on the raw grid ────────────────────
    z_10_asc  = z_10[::-1]
    re_10_asc = re_10[::-1]
    re_recon_asc = np.interp(z_raw_asc if PLOT_SPACE == 'z' else z_raw_asc,
                             z_10_asc, re_10_asc)
    re_recon = re_recon_asc[::-1]

    z_tau_asc  = z_tau[::-1]
    re_tau_asc = re_tau[::-1]
    re_recon_tau_asc = np.interp(z_raw_asc, z_tau_asc, re_tau_asc)
    re_recon_tau = re_recon_tau_asc[::-1]

    ax.plot(re_raw,       y_raw,      color='0.55', linewidth=1.2,
            marker='o', markersize=2.5, label='Raw in-situ')
    ax.plot(re_10,        y_10,       color='steelblue', linewidth=1.8,
            marker='o', markersize=5,   label='Even-z interp.')
    ax.plot(re_tau,       y_tau_plot, color='darkorange', linewidth=1.8,
            marker='s', markersize=5,   linestyle='--', label='τ-weighted interp.')
    ax.plot(re_recon,     y_raw,      color='steelblue', linewidth=1.0,
            linestyle=':', alpha=0.7, label='Even-z resampled')
    ax.fill_betweenx(y_raw, re_raw, re_recon,
                     alpha=0.12, color='steelblue')
    ax.plot(re_recon_tau, y_raw,      color='darkorange', linewidth=1.0,
            linestyle=':', alpha=0.7, label='τ-weighted resampled')
    ax.fill_betweenx(y_raw, re_raw, re_recon_tau,
                     alpha=0.12, color='darkorange')

    if invert_y:
        ax.invert_yaxis()

    ax.set_title(f'Profile {idx+1}  |  {n_levels_each[idx]} lev  |  '
                 f'RMSE z={rmse_all[idx]:.2f} τ={rmse_tau_all[idx]:.2f} μm',
                 fontsize=7.5)
    ax.set_xlabel('$r_e$ (μm)', fontsize=8)
    ax.set_ylabel(ylabel, fontsize=8)
    ax.tick_params(labelsize=7)

axes[0].legend(fontsize=6, loc='best')

for ax in axes[N_WORST:]:
    ax.set_visible(False)

space_label = 'altitude' if PLOT_SPACE == 'z' else 'normalized optical depth τ̃'
fig.suptitle(f'Worst-case even-z interpolation errors with τ-weighted comparison\n'
             f'(y-axis: {space_label})',
             fontsize=10, y=1.01)
fig.tight_layout()
plt.show()


## 6. Mean profile and variability

Mean ± 1σ envelope for both the raw profiles (interpolated to a common normalized grid for averaging)
and the 10-level profiles. Shows whether the 10-level representation preserves mean behavior and spread.

In [ ]:
# Project all profiles onto a common 100-point normalized altitude grid
# (0 = cloud base, 1 = cloud top) for meaningful cross-profile statistics
N_COMMON = 100
z_common = np.linspace(0, 1, N_COMMON)

raw_on_common   = np.zeros((n_profiles, N_COMMON))
interp_on_common = np.zeros((n_profiles, N_COMMON))
tau_on_common   = np.zeros((n_profiles, N_COMMON))

for i in range(n_profiles):
    # Raw profile
    z = altitudes_raw[i]
    z_frac_asc = ((z - z[-1]) / (z[0] - z[-1]))[::-1]
    re_asc = profiles_raw[i][::-1]
    raw_on_common[i] = np.interp(z_common, z_frac_asc, re_asc)

    # Even-z interpolation
    z10 = altitudes_10[i]
    z10_frac_asc = ((z10 - z10[-1]) / (z10[0] - z10[-1]))[::-1]
    re10_asc = profiles_10[i][::-1]
    interp_on_common[i] = np.interp(z_common, z10_frac_asc, re10_asc)

    # τ-weighted interpolation
    z_tau = altitudes_tau[i]
    z_tau_frac_asc = ((z_tau - z_tau[-1]) / (z_tau[0] - z_tau[-1]))[::-1]
    re_tau_asc = profiles_tau[i][::-1]
    tau_on_common[i] = np.interp(z_common, z_tau_frac_asc, re_tau_asc)

raw_mean    = raw_on_common.mean(axis=0)
raw_std     = raw_on_common.std(axis=0)
interp_mean = interp_on_common.mean(axis=0)
interp_std  = interp_on_common.std(axis=0)
tau_mean    = tau_on_common.mean(axis=0)
tau_std     = tau_on_common.std(axis=0)

fig, axes = plt.subplots(1, 3, figsize=(14, 5), sharey=True)

for ax, mean, std, label, color in [
    (axes[0], raw_mean,    raw_std,    'Raw in-situ',        'steelblue'),
    (axes[1], interp_mean, interp_std, '10-level (even z)',  'steelblue'),
    (axes[2], tau_mean,    tau_std,    '10-level (τ-weighted)', 'darkorange'),
]:
    ax.fill_betweenx(z_common, mean - std, mean + std,
                     alpha=0.25, color=color, label='±1σ')
    ax.plot(mean, z_common, color=color, linewidth=2, label='Mean')
    ax.set_xlabel('$r_e$ (μm)')
    ax.set_ylabel('Normalized altitude (0 = base, 1 = top)')
    ax.set_title(f'{label}\nMean ± 1σ across all {n_profiles} profiles')
    ax.legend()

# Overlay all three means on the left panel for direct comparison
axes[0].plot(interp_mean, z_common, color='steelblue', linewidth=1.5,
             linestyle='--', alpha=0.7, label='Even-z mean')
axes[0].plot(tau_mean, z_common, color='darkorange', linewidth=1.5,
             linestyle='--', alpha=0.7, label='τ-weighted mean')
axes[0].legend(fontsize=7)

fig.suptitle('Mean profile structure: raw vs. even-z vs. τ-weighted', fontsize=10)
fig.tight_layout()
plt.show()

# Quantify bias vs. raw
bias_even = interp_mean - raw_mean
bias_tau  = tau_mean    - raw_mean
print(f'Mean bias vs. raw (on normalized altitude grid):')
print(f'  {"Method":<28} {"Max |bias|":>12} {"Mean |bias|":>12}')
print(f'  {"-"*54}')
print(f'  {"Even-z":<28} {np.abs(bias_even).max():>12.3f} {np.abs(bias_even).mean():>12.3f} μm')
print(f'  {"τ-weighted":<28} {np.abs(bias_tau).max():>12.3f} {np.abs(bias_tau).mean():>12.3f} μm')
